In [ ]:
import numpy as np
from torch.utils.data import DataLoader, Dataset, Sampler

In [ ]:
class DummyPatchSpecs:
    data_index: int

In [ ]:
class DummyImageStack:

    def __init__(self, file_id: int):
        self.file_id = file_id
        self.is_loaded: bool = False
        self.extract_patch_calls: int = 0

    def extract_patch(self):
        self.extract_patch_calls += 1
        if not self.is_loaded:
            self._load()
        patch = f"Patch from image stack {self.file_id}"
        return patch

    def _load(self):
        self.is_loaded = True
        print(f"Loaded file: {self.file_id}")

    def close(self):
        # TODO: raise error if not loaded?
        self.is_loaded = False
        self.extract_patch_calls = 0
        print(f"Closed file: {self.file_id}")

In [ ]:
class DummyPatchingStrategy:

    def __init__(self, image_stacks: list[DummyImageStack]):
        self.n_patches_per_image_stack = [
            np.random.randint(16, 32) for _ in image_stacks
        ]
        self.patch_bins = np.concatenate(
            [[0], np.cumsum(self.n_patches_per_image_stack)]
        )

    def __getitem__(self, index: int) -> DummyPatchSpecs:
        # minus 1, because digitize returns 1 for first bin
        data_index = np.digitize(index, self.patch_bins) - 1
        return {"data_index": data_index}

In [ ]:
class DummyDataset(Dataset):

    def __init__(self, image_stacks: list[DummyImageStack]):
        self.image_stacks = image_stacks
        self.patching_strategy = DummyPatchingStrategy(image_stacks)

    def __len__(self):
        return sum(self.patching_strategy.n_patches_per_image_stack)

    def __getitem__(self, index):
        patch_specs = self.patching_strategy[index]
        data_index = patch_specs["data_index"]
        patch = self.image_stacks[data_index].extract_patch()
        return patch

In [ ]:
class FileLoadingSampler(Sampler):

    def __init__(self, dataset: DummyDataset, n_files_loaded: int):
        self.dataset = dataset
        self.n_files_loaded = n_files_loaded

        self.n_files = len(dataset.image_stacks)

    def __iter__(self):
        self.cleanup()
        file_indices = np.arange(self.n_files)
        np.random.shuffle(file_indices)
        file_groups = [
            file_indices[i : i + self.n_files_loaded]
            for i in range(0, self.n_files, self.n_files_loaded)
        ]
        for file_group in file_groups:
            patch_indices = np.zeros((0,))
            for file_index in file_group:
                file_patch_indices = np.arange(
                    self.dataset.patching_strategy.patch_bins[file_index],
                    self.dataset.patching_strategy.patch_bins[file_index + 1],
                )
                patch_indices = np.concatenate([patch_indices, file_patch_indices])

            np.random.shuffle(patch_indices)
            for patch_index in patch_indices:
                yield patch_index
                self.cleanup()

    def cleanup(self):
        for image_stack, n_patches in zip(
            self.dataset.image_stacks,
            self.dataset.patching_strategy.n_patches_per_image_stack,
        ):
            if image_stack.extract_patch_calls >= n_patches:
                image_stack.close()

In [ ]:
image_stacks = [DummyImageStack(i) for i in range(10)]
dataset = DummyDataset(image_stacks)

In [ ]:
dataloader = DataLoader(
    dataset=dataset, sampler=FileLoadingSampler(dataset, 3), batch_size=4
)

In [ ]:
np.diff(dataset.patching_strategy.patch_bins)

In [ ]:
track_files_open = []
files_open = sum(image_stack.is_loaded for image_stack in dataset.image_stacks)
print("--- files open: ", files_open)
for patch in dataloader:
    files_open = sum(image_stack.is_loaded for image_stack in dataset.image_stacks)
    track_files_open.append(files_open)
    print("--- files open: ", files_open)
    print(patch)
print(track_files_open)

In [ ]:
for i, image_stack in enumerate(dataset.image_stacks):
    print(i, image_stack.is_loaded)

In [ ]:
class Manager:
    def notify(self, class_name, method_name):
        print(f"Manager notified: {class_name}.{method_name} was called.")

    def register(self, cls, method_name):
        method = getattr(cls, method_name)

        def wrapper(self_, *args, **kwargs):
            self.notify(cls.__name__, method.__name__)
            return method(self_, *args, **kwargs)

        setattr(cls, method_name, wrapper)  # Replace method with wrapped version

# Example: Existing class (already defined)
class ExampleClass:
    def some_method(self):
        print("Executing some_method")

    def another_method(self):
        print("Executing another_method")

# Apply the wrapper dynamically
manager = Manager()
manager.register(ExampleClass, "some_method")
# manager.register(ExampleClass, "another_method")

# Test
obj = ExampleClass()
obj.some_method()  # Manager gets notified
obj.another_method()  # Manager gets notified
